[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vinod-seth/slm-development/blob/main/tutorial/01_small_language_models_what_they_are_and_why_they_matter/01_slms_vs_llms.ipynb)

# SLMs vs LLMs: Scope, Trade-offs, and Real Use Cases

| | |
|---|---|
| **Domain** | GenAI |
| **Module** | Small Language Models: What They Are and Why They Matter |
| **Difficulty** | Beginner |
| **Estimated Time** | 25 minutes |
| **Prerequisites** | Basic Python programming knowledge; familiarity with what a model is and the difference between training and inference; no prior deep learning or NLP experience required |

---

## 📍 Lesson Roadmap

- **Core Concepts** — Understand what separates an SLM from an LLM using parameter count, hardware tiers, and deployment context.
- **Model Families** — Meet the four SLM families you will work with throughout this course: SmolLM2, Phi-2, TinyLlama, and DistilGPT-2.
- **Technical Deep-Dive** — Run your first SLM inference in under ten lines of Python using Hugging Face Transformers.
- **Hands-On Exercise** — Load two different SLMs, compare their output latency, and record your observations.
- **Concept Check** — Verify your understanding with three targeted questions before moving to Lesson 2.

---

## Learning Objectives

By the end of this lesson, you will be able to:

- Distinguish small language models from large language models using parameter count, hardware requirements, and deployment context.
- Identify three categories of real-world tasks where SLMs outperform LLMs on latency, privacy, or domain-scope grounds.
- Name at least four SLM model families covered in this course (SmolLM2, Phi-2, TinyLlama, DistilGPT-2) and their approximate sizes.

---

## ⚙️ Setup — Run This Cell First

This cell installs all required dependencies. It runs silently and takes about 30–60 seconds on Colab.

In [ ]:
# ── Auto-install dependencies ──────────────────────────────────────────────
# This cell installs everything you need. No manual setup required.
# Pinned versions ensure reproducibility (last verified: 2025-06).
# ───────────────────────────────────────────────────────────────
!pip install -q transformers==4.41.2 torch==2.3.0 accelerate==0.30.1 sentencepiece==0.2.0

# Verify installation succeeded
import transformers, torch
print(f"✅ transformers {transformers.__version__}")
print(f"✅ torch {torch.__version__}")
print(f"✅ CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
else:
    print("   Running on CPU — all code in this notebook works on CPU.")
print("\n✅ Setup complete! You can now run every cell below.")

---

## 🟢 Core Concepts

### What Makes a Model "Small"?

There is no universally ratified threshold, but the field treats models with **under ~3 billion parameters** as small language models (SLMs). Most practical SLMs sit between 100 million and 1.5 billion parameters. Large language models (LLMs) typically start at 7 billion parameters and extend into the hundreds of billions.

Parameters are the numeric weights a model learns during training. More parameters generally means broader world knowledge, stronger multi-step reasoning, and better performance on open-ended tasks — but the cost scales sharply.

```
Parameter count and its hardware implications:

┌─────────────────────────┬──────────────────┬────────────────────────────┐
│ Model Class             │ Parameter Range  │ Minimum GPU VRAM (FP16)    │
├─────────────────────────┼──────────────────┼────────────────────────────┤
│ Micro SLM               │ 100M – 350M      │ CPU or 2 GB GPU            │
│ Small SLM               │ 350M – 1.5B      │ 4 GB GPU                   │
│ Mid SLM                 │ 1.5B – 3B        │ 6–8 GB GPU                 │
│ Standard LLM            │ 7B – 13B         │ 16–24 GB GPU               │
│ Large LLM               │ 70B+             │ Multi-GPU or cloud cluster  │
└─────────────────────────┴──────────────────┴────────────────────────────┘
```

> **Note:** These VRAM estimates assume full FP16 precision. 4-bit quantization roughly halves the footprint — covered in Module 5.

### The Capability Trade-off Is Not Linear

A model with 1/70th the parameters of GPT-4-class LLMs does not deliver 1/70th the capability. For **narrow, well-defined tasks** — classifying support tickets, extracting structured fields from medical forms, generating short product descriptions — a well-trained 360M-parameter model routinely matches or beats a general-purpose 70B model. The LLM's extra capacity is simply unused.

Brown et al. (2020) *Language Models are Few-Shot Learners* established that scale unlocks emergent capabilities for broad, open-ended tasks. The inverse insight for practitioners: if your task is not broad and open-ended, paying for scale is waste.

### Three Decision Axes: Latency, Privacy, Domain Scope

**Latency** — A 135M-parameter model generates a token in under 5 ms on commodity hardware. A 70B model requires tens of milliseconds per token on an A100 cluster. For real-time autocomplete, voice interfaces, or edge devices, that gap is disqualifying.

**Privacy** — Running an SLM locally means patient records, legal documents, and financial data never leave the premises. Regulatory frameworks like HIPAA and GDPR make this constraint non-negotiable for many organizations.

**Domain Scope** — A model fine-tuned on a specific corpus (e.g., industrial sensor logs) builds dense representations of that domain with far less data than an LLM would need to generalize from scratch. Narrow scope is a feature, not a limitation.

### The Four Model Families in This Course

| Model Family | Developer | Approx. Size | Key Characteristic |
|---|---|---|---|
| **SmolLM2** | Hugging Face | 135M – 1.7B | Modern architecture; strong on instruction following at small scale (2024) |
| **Phi-2** | Microsoft | 2.7B | Trained on high-quality synthetic data; matches models twice its size on HumanEval and BIG-Bench Hard reasoning benchmarks |
| **TinyLlama** | Zhang et al. | 1.1B | Llama 2 architecture; trained on 3T tokens; strong community support |
| **DistilGPT-2** | Hugging Face | 82M | Distilled from GPT-2; ideal for constrained-hardware experiments |

> **Important:** SmolLM2 is the primary model family for this course. It represents the current state of the art for sub-2B open models as of 2024 — a category absent from most competing curricula.
>
> *Note: Phi-2 has been succeeded by Phi-3-mini (3.8B) and Phi-3.5-mini. We retain Phi-2 for this course because it is stable, widely documented, and fits the sub-3B SLM definition.*

---

## 🔷 Technical Deep-Dive

### Running Your First SLM Inference

The following code loads SmolLM2-135M (Hugging Face, 2024) and runs a text generation task. It intentionally keeps the structure minimal so you can see the full pipeline without noise.

**Just click the ▶️ play button on the cell below** — the model downloads automatically (~270 MB on first run).

In [ ]:
"""
Your first SLM inference!

This cell loads SmolLM2-135M and generates a short text completion.
It works on both CPU and GPU — no configuration needed.

Expected run time:
  - CPU: ~5–10 seconds for first load, <1 second per generation
  - GPU: ~2–3 seconds for first load, <0.1 seconds per generation
"""

import time
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# ── Configuration ──────────────────────────────────────────────────
MODEL_ID = "HuggingFaceTB/SmolLM2-135M-Instruct"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MAX_NEW_TOKENS = 80
TEMPERATURE = 0.7


def load_model(model_id: str, device: str):
    """Load tokenizer and model onto the target device."""
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        torch_dtype=torch.float16 if device == "cuda" else torch.float32,
    )
    model.to(device)
    model.eval()
    return tokenizer, model


def generate_completion(
    prompt: str,
    tokenizer,
    model,
    device: str,
    max_new_tokens: int = MAX_NEW_TOKENS,
    temperature: float = TEMPERATURE,
) -> tuple:
    """
    Generate a text completion and return the output with wall-clock latency.

    Returns:
        tuple: (generated_text, latency_in_seconds)
    """
    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    start_time = time.perf_counter()

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            do_sample=True,
            top_p=0.9,                # Nucleus sampling for quality
            repetition_penalty=1.1,   # Prevents repetition loops in small models
            pad_token_id=tokenizer.eos_token_id,
        )

    latency = time.perf_counter() - start_time

    # Decode only the newly generated tokens, not the prompt
    generated_tokens = output_ids[0][inputs["input_ids"].shape[-1]:]
    generated_text = tokenizer.decode(generated_tokens, skip_special_tokens=True)

    return generated_text, latency


# ── Run inference ──────────────────────────────────────────────────
print(f"Loading {MODEL_ID} on {DEVICE}...")
tokenizer, model = load_model(MODEL_ID, DEVICE)

prompt = (
    "Summarize the key difference between a small language model "
    "and a large language model in two sentences:"
)

output, latency_secs = generate_completion(prompt, tokenizer, model, DEVICE)

print(f"\nPrompt:  {prompt}")
print(f"Output:  {output}")
print(f"Latency: {latency_secs:.3f} seconds on {DEVICE}")

**Expected output (your text will differ due to temperature-based sampling):**

```
Loading HuggingFaceTB/SmolLM2-135M-Instruct on cpu...

Prompt:  Summarize the key difference between a small language model and a large language model in two sentences:
Output:  Small language models have fewer parameters and run efficiently on consumer hardware, making them well-suited for specific, narrow tasks. Large language models trade computational cost for broader reasoning and knowledge across diverse domains.
Latency: 0.612 seconds on cpu
```

> **Note:** Your output text will differ on each run due to temperature-based sampling. Latency varies by CPU model. On a CUDA GPU, expect under 0.1 seconds.

### Comparing Parameter Counts Programmatically

Use this utility to inspect any Hugging Face model's parameter count before committing to download:

In [ ]:
"""
Parameter counter utility.

Reports trainable and total parameter counts for any HF model.
Useful for verifying model size before a full download.
"""

from transformers import AutoModelForCausalLM
import torch


def count_parameters(model_id: str) -> dict:
    """
    Return trainable and total parameter counts for a causal LM.

    Args:
        model_id: Hugging Face model identifier string.

    Returns:
        Dictionary with 'trainable' and 'total' keys.
    """
    model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=torch.float32)
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    return {"trainable": trainable, "total": total}


# Compare two SLM families
model_ids = [
    "distilgpt2",                          # ~82M
    "HuggingFaceTB/SmolLM2-135M-Instruct", # ~135M
]

for mid in model_ids:
    counts = count_parameters(mid)
    print(
        f"{mid}: "
        f"{counts['total']:,} total params "
        f"({counts['total'] / 1e6:.1f}M)"
    )

**Expected output:**

```
distilgpt2: 81,912,576 total params (81.9M)
HuggingFaceTB/SmolLM2-135M-Instruct: 134,515,712 total params (134.5M)
```

> **Important:** Loading full-precision models on CPU consumes roughly 4 bytes per parameter. A 135M model uses ~540 MB of RAM — well within range of any modern laptop.

---

## 🧪 Hands-On Exercise

**Goal:** Run inference with two different SLM families side-by-side and observe the latency difference on your machine.

**Time estimate:** 10–12 minutes

### Step 1 — Confirm your environment

Run the cell below. You should see version numbers without errors.

In [ ]:
# Step 1: Confirm your environment
import transformers, torch
print(f"transformers: {transformers.__version__}")
print(f"torch: {torch.__version__}")
print("✅ Environment ready!")

### Step 2 — Run inference with SmolLM2-135M

The model was already loaded above, so this cell reuses it. Record the latency.

In [ ]:
# Step 2: Inference with SmolLM2-135M (already loaded above)
exercise_prompt = "Explain why edge devices benefit from smaller AI models:"

smolm2_output, smolm2_latency = generate_completion(
    exercise_prompt, tokenizer, model, DEVICE
)

print(f"Model:   SmolLM2-135M-Instruct")
print(f"Output:  {smolm2_output}")
print(f"Latency: {smolm2_latency:.3f} seconds")

### Step 3 — Swap the model to DistilGPT-2 and run again

In [ ]:
# Step 3: Load DistilGPT-2 and run the same prompt
print("Loading distilgpt2...")
tokenizer_dgpt2, model_dgpt2 = load_model("distilgpt2", DEVICE)

dgpt2_output, dgpt2_latency = generate_completion(
    exercise_prompt, tokenizer_dgpt2, model_dgpt2, DEVICE
)

print(f"\nModel:   DistilGPT-2")
print(f"Output:  {dgpt2_output}")
print(f"Latency: {dgpt2_latency:.3f} seconds")

> **Note:** DistilGPT-2 has no instruction-following fine-tuning, so its raw completions will read differently from SmolLM2. You will explore why fine-tuning matters in Module 4.

### Step 4 — Document your findings

In [ ]:
# Step 4: Auto-generate your results summary
params_smolm2 = count_parameters("HuggingFaceTB/SmolLM2-135M-Instruct")
params_dgpt2 = count_parameters("distilgpt2")

results = f"""
# Exercise 1 Results

| Model | Parameters | Latency (seconds) | Hardware |
|---|---|---|---|
| SmolLM2-135M-Instruct | {params_smolm2['total']/1e6:.1f}M | {smolm2_latency:.3f} | {DEVICE} |
| DistilGPT-2 | {params_dgpt2['total']/1e6:.1f}M | {dgpt2_latency:.3f} | {DEVICE} |

## Observation
{'SmolLM2' if smolm2_latency > dgpt2_latency else 'DistilGPT-2'} was slower, 
which {'is expected since it has more parameters' if smolm2_latency > dgpt2_latency else 'may be due to model architecture differences'}.
"""

print(results)
print("✅ Exercise complete! Your results are shown above.")

---

## ❓ Concept Check

### Question 1

A healthcare startup needs to run text classification on patient intake forms. Their requirements are: no data leaves the hospital network, inference must complete in under 200 ms on a standard workstation, and the task is limited to 12 predefined symptom categories. Which class of model is most appropriate?

- **A)** A fine-tuned SLM in the 135M–360M parameter range, deployed locally
- **B)** A general-purpose 70B LLM accessed via a cloud API
- **C)** A 7B LLM fine-tuned on-premises with full precision
- **D)** DistilBERT without fine-tuning, used zero-shot

<details>
<summary>🔑 Click to Reveal Answer & Explanation</summary>

**Correct Answer:** Option A.

**Explanation:**
The task is narrow (12 categories), latency-sensitive (200 ms budget), and privacy-constrained (no external data transfer). These three conditions collectively rule out cloud APIs regardless of model quality. A fine-tuned SLM in the 135M–360M range runs comfortably on CPU within the latency budget and keeps all data local. A 7B model running full precision on-premises would likely exceed the 200 ms constraint and require significantly more VRAM.

</details>

### Question 2

Which statement about the relationship between parameter count and task performance is most accurate?

- **A)** More parameters always produce better results for any task.
- **B)** SLMs always underperform LLMs because they have seen less training data.
- **C)** For narrow, well-defined tasks, a fine-tuned SLM can match or exceed a general-purpose LLM.
- **D)** Parameter count has no relationship to hardware requirements.

<details>
<summary>🔑 Click to Reveal Answer & Explanation</summary>

**Correct Answer:** Option C.

**Explanation:**
Brown et al. (2020) demonstrated that scale unlocks emergent capability for *broad* tasks. However, when the task scope is narrow and well-defined, the LLM's extra capacity goes unused. A domain-specific fine-tuned SLM builds dense representations of the target domain and routinely matches much larger general-purpose models on classification, extraction, and constrained generation tasks.

</details>

### Question 3 — Reflection Prompt

Describe a real project or professional context where you would choose an SLM over an LLM. Justify your choice using at least two of the three decision axes: latency, privacy, and domain scope.

*(There is no single correct answer. Use the criteria below to evaluate your own response.)*

<details>
<summary>🔑 Click to Reveal Evaluation Criteria</summary>

**Strong response includes:**
- A specific, concrete use case (not a generic one like "a chatbot").
- Explicit reference to at least two decision axes with quantitative or regulatory reasoning.
- An acknowledgment of what the SLM trade-off costs — what capability are you giving up?

**Example of a strong response:**
"A quality-control system on a factory floor that classifies whether a machine’s vibration signature indicates normal operation or one of five fault conditions. Latency: the controller must flag anomalies in under 50 ms. Privacy: sensor data is proprietary process IP. Domain scope: only six output classes are needed. Trade-off accepted: the model cannot answer open-ended questions, but it never needs to."

</details>

---

## Summary

- **SLMs occupy the sub-3B parameter range.** They require dramatically less hardware and power than LLMs, making local and edge deployment feasible on consumer GPUs or even CPUs.
- **The capability trade-off is task-dependent, not absolute.** SLMs match or beat LLMs on narrow, well-defined tasks — the three primary reasons to choose an SLM are latency constraints, privacy requirements, and domain-scope boundaries.
- **Four model families anchor this course.** SmolLM2 (135M–1.7B), Phi-2 (2.7B), TinyLlama (1.1B), and DistilGPT-2 (82M) span the hardware feasibility spectrum from a Raspberry Pi to a workstation GPU.

---

## References & Credits

- Brown et al. (2020) *Language Models are Few-Shot Learners*. [arXiv:2005.14165](https://arxiv.org/abs/2005.14165)
- Hugging Face. *SmolLM2 Model Card*. [HuggingFaceTB/SmolLM2-135M-Instruct](https://huggingface.co/HuggingFaceTB/SmolLM2-135M-Instruct) — Last verified: 2024-11
- Microsoft Research. *Phi-2 Model Card*. [microsoft/phi-2](https://huggingface.co/microsoft/phi-2) — Last verified: 2024-11
- Zhang et al. (2024) *TinyLlama: An Open-Source Small Language Model*. [arXiv:2401.02385](https://arxiv.org/abs/2401.02385)
- Hugging Face. *DistilGPT-2 Model Card*. [distilgpt2](https://huggingface.co/distilgpt2) — Last verified: 2024-11
- Sanh et al. (2019) *DistilBERT, a distilled version of BERT*. [arXiv:1910.01108](https://arxiv.org/abs/1910.01108)